In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window

storage_account = "annagaplanyandatabricks"
container = "pdb-data"

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/pdb_lab"
silver_path = f"{base_path}/silver"

@dp.materialized_view(
    name="silver_entity_mv",
    path=f"{silver_path}/silver_entity_mv"
)
def silver_entity_mv():
    df = spark.read.table("pdb_lab.bronze_entity")

    w = Window.partitionBy(
        F.lower(F.col("pdb_code")),
        F.col("entity_id")
    ).orderBy(F.col("ingested_at").desc())

    return (
        df.select(
            "run_id",
            F.lower(F.col("pdb_code")).alias("pdb_code"),
            "source_url",
            "ingested_at",
            "entity_id",
            "details",
            F.col("formula_weight").cast("double").alias("formula_weight"),
            "pdbx_description",
            "pdbx_ec",
            "pdbx_fragment",
            "pdbx_mutation",
            F.col("pdbx_number_of_molecules").cast("int").alias("pdbx_number_of_molecules"),
            "src_method",
            "entity_type",
        )
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

@dp.materialized_view(
    name="silver_exptl_mv",
    path=f"{silver_path}/silver_exptl_mv"
)
@dp.expect(
    "valid_exptl_method",
    """
    upper(trim(method)) IN (
      'ELECTRON CRYSTALLOGRAPHY',
      'ELECTRON MICROSCOPY',
      'EPR',
      'FIBER DIFFRACTION',
      'FLUORESCENCE TRANSFER',
      'INFRARED SPECTROSCOPY',
      'NEUTRON DIFFRACTION',
      'POWDER DIFFRACTION',
      'SOLID-STATE NMR',
      'SOLUTION NMR',
      'SOLUTION SCATTERING',
      'THEORETICAL MODEL',
      'X-RAY DIFFRACTION'
    )
    """
)
def silver_exptl_mv():
    df = spark.read.table("pdb_lab.bronze_exptl")

    w = Window.partitionBy(
        F.lower(F.col("pdb_code")),
        F.col("entry_id")
    ).orderBy(F.col("ingested_at").desc())

    return (
        df.select(
            "run_id",
            F.lower(F.col("pdb_code")).alias("pdb_code"),
            "source_url",
            "ingested_at",
            "entry_id",
            F.trim(F.col("method")).alias("method"),
        )
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

@dp.materialized_view(
    name="silver_entity_poly_seq_mv",
    path=f"{silver_path}/silver_entity_poly_seq_mv"
)
def silver_entity_poly_seq_mv():
    df = spark.read.table("pdb_lab.bronze_entity_poly_seq")

    seq_num_int = F.col("seq_num").cast("int")

    w = Window.partitionBy(
        F.lower(F.col("pdb_code")),
        F.col("entity_id"),
        F.col("mon_id"),
        seq_num_int
    ).orderBy(F.col("ingested_at").desc())

    return (
        df.select(
            "run_id",
            F.lower(F.col("pdb_code")).alias("pdb_code"),
            "source_url",
            "ingested_at",
            "entity_id",
            "mon_id",
            seq_num_int.alias("seq_num"),
            "hetero",
        )
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

@dp.materialized_view(
    name="silver_chem_comp_mv",
    path=f"{silver_path}/silver_chem_comp_mv"
)
def silver_chem_comp_mv():
    df = spark.read.table("pdb_lab.bronze_chem_comp")

    w = Window.partitionBy(
        F.lower(F.col("pdb_code")),
        F.col("chem_comp_id")
    ).orderBy(F.col("ingested_at").desc())

    return (
        df.select(
            "run_id",
            F.lower(F.col("pdb_code")).alias("pdb_code"),
            "source_url",
            "ingested_at",
            "chem_comp_id",
            "formula",
            F.col("formula_weight").cast("double").alias("formula_weight"),
            "model_details",
            "model_erf",
            "model_source",
            "mon_nstd_class",
            "mon_nstd_details",
            "mon_nstd_flag",
            "mon_nstd_parent",
            "mon_nstd_parent_comp_id",
            "name",
            F.col("number_atoms_all").cast("int").alias("number_atoms_all"),
            F.col("number_atoms_nh").cast("int").alias("number_atoms_nh"),
            "one_letter_code",
            "pdbx_ambiguous_flag",
            "pdbx_casnum",
            "pdbx_class_1",
            "pdbx_class_2",
            "pdbx_comp_type",
            "pdbx_component_no",
            F.col("pdbx_formal_charge").cast("int").alias("pdbx_formal_charge"),
            "pdbx_ideal_coordinates_details",
            "pdbx_ideal_coordinates_missing_flag",
            F.to_date("pdbx_initial_date").alias("pdbx_initial_date"),
            "pdbx_model_coordinates_db_code",
            "pdbx_model_coordinates_details",
            "pdbx_model_coordinates_missing_flag",
            "pdbx_modification_details",
            F.to_date("pdbx_modified_date").alias("pdbx_modified_date"),
            "pdbx_nscnum",
            F.col("pdbx_number_subcomponents").cast("int").alias("pdbx_number_subcomponents"),
            "pdbx_pcm",
            "pdbx_processing_site",
            "pdbx_release_status",
            "pdbx_replaced_by",
            "pdbx_replaces",
            "pdbx_reserved_name",
            "pdbx_smiles",
            "pdbx_status",
            "pdbx_subcomponent_list",
            "pdbx_synonyms",
            "pdbx_type",
            "pdbx_type_modified",
            "three_letter_code",
            "chem_comp_type",
        )
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

@dp.materialized_view(
    name="silver_pdbx_database_pdb_obs_spr_mv",
    path=f"{silver_path}/silver_pdbx_database_pdb_obs_spr_mv"
)
def silver_pdbx_database_pdb_obs_spr_mv():
    df = spark.read.table("pdb_lab.bronze_pdbx_database_pdb_obs_spr")

    w = Window.partitionBy(
        F.lower(F.col("pdb_code")),
        F.col("obs_id")
    ).orderBy(F.col("ingested_at").desc())

    return (
        df.select(
            "run_id",
            F.lower(F.col("pdb_code")).alias("pdb_code"),
            "source_url",
            "ingested_at",
            "obs_pdb_id",
            "replace_pdb_id",
            F.to_date("obs_date").alias("obs_date"),
            "details",
            "obs_id",
        )
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )